# Telemetry Anomaly Detection Analysis

This notebook provides tools for analyzing spacecraft telemetry anomaly detection datasets.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from visualize import TelemetryVisualizer

## Initialize Visualizer

In [ ]:
visualizer = TelemetryVisualizer(output_dir='../plots')
data_dir = Path('../data')

## Check Available Data

In [ ]:
print('Available files in data directory:')
for path in sorted(data_dir.rglob('*')):
    if path.is_file():
        print(f'  {path.relative_to(data_dir)}')

## Load NASA SMAP & MSL Data

Place your downloaded data in the `data/` folder:
- `data/train/` - Training .npy files
- `data/test/` - Test .npy files  
- `data/labeled_anomalies.csv` - Anomaly labels

In [ ]:
# Check if data exists
train_dir = data_dir / 'train'
test_dir = data_dir / 'test'

if train_dir.exists() and test_dir.exists():
    data = visualizer.load_nasa_smap_msl(data_dir)
    print(f'Loaded {len(data["train"])} training channels')
    print(f'Loaded {len(data["test"])} test channels')
    
    if data['labels'] is not None:
        print(f'Loaded {len(data["labels"])} anomaly labels')
else:
    print('Please download NASA SMAP & MSL data to the data/ folder')
    print('Download from: https://www.kaggle.com/datasets/patrickfleith/nasa-anomaly-detection-dataset-smap-msl')

## Visualize Channel Data

In [ ]:
if 'data' in locals() and data['train']:
    # Get first channel
    channel_id = list(data['train'].keys())[0]
    print(f'Visualizing channel: {channel_id}')
    
    # Plot channel overview
    visualizer.plot_channel_overview(data, channel_id, data['labels'])
else:
    print('No data loaded. Please add data to the data/ folder first.')

## Plot Multiple Channels

In [ ]:
if 'data' in locals() and data['train']:
    # Get first 5 channels
    channel_ids = list(data['train'].keys())[:5]
    
    # Plot multiple channels
    visualizer.plot_multiple_channels(data, channel_ids)
else:
    print('No data loaded.')

## Anomaly Statistics

In [ ]:
if 'data' in locals() and data['labels'] is not None:
    visualizer.plot_anomaly_statistics(data['labels'])
else:
    print('No label data loaded.')

## Correlation Analysis

In [ ]:
if 'data' in locals() and data['train']:
    # Plot correlation matrix for first 10 channels
    visualizer.plot_correlation_matrix(data, max_channels=10)
else:
    print('No data loaded.')

## Time Series Feature Analysis

In [ ]:
if 'data' in locals() and data['train']:
    # Analyze first channel
    channel_id = list(data['train'].keys())[0]
    visualizer.plot_time_series_features(data, channel_id)
else:
    print('No data loaded.')

## Generate Summary Report

In [ ]:
if 'data' in locals():
    visualizer.generate_summary_report(data, data.get('labels'))
else:
    print('No data loaded.')

## Load CSV Data (for other datasets)

In [ ]:
# Find all CSV files in data directory
csv_files = list(data_dir.glob('*.csv'))

if csv_files:
    print('Found CSV files:')
    for csv_file in csv_files:
        print(f'\nLoading {csv_file.name}...')
        df = pd.read_csv(csv_file)
        print(f'  Shape: {df.shape}')
        print(f'  Columns: {list(df.columns)}')
        print(f'  First few rows:')
        display(df.head())
else:
    print('No CSV files found in data directory.')

In [ ]:
# Quick visualization of CSV data
if csv_files:
    for csv_file in csv_files:
        df = pd.read_csv(csv_file)
        
        # Get numeric columns
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        
        if len(numeric_cols) > 0:
            fig, axes = plt.subplots(min(3, len(numeric_cols)), 1, figsize=(14, 4*min(3, len(numeric_cols))))
            if len(numeric_cols) == 1:
                axes = [axes]
            
            for i, col in enumerate(numeric_cols[:3]):
                axes[i].plot(df[col].values, alpha=0.8)
                axes[i].set_title(f'{csv_file.stem} - {col}', fontsize=10)
                axes[i].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()